In [1]:
!curl -o car_fuel_efficiency_2026.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  620k  100  620k    0     0   575k      0  0:00:01  0:00:01 --:--:--  575k


In [29]:
import pandas as pd
import numpy as np
df=pd.read_csv('car_fuel_efficiency_2026.csv')
df.columns=df.columns.str.lower().str.replace(' ','_')
strings=list(df.dtypes[df.dtypes=='object'].index)
for col in strings:
    df[col]=df[col].str.lower().str.replace(' ','_')
y=np.log1p(df['fuel_efficiency_mpg'].values)
y=np.array(y)
#del df['fuel_efficiency_mpg']
base=['engine_displacement','horsepower','vehicle_weight','model_year']
n=len(df)
n_val=int(n*0.2)
n_test=int(n*0.2)
n_train=n-n_val-n_test
np.random.seed(42)
idx=np.arange(n)
np.random.shuffle(idx)
df_train=df.iloc[idx[:n_train]].reset_index(drop=True)
df_val=df.iloc[idx[n_train:n_train+n_val]].reset_index(drop=True)
df_test=df.iloc[idx[n_train+n_val]].reset_index(drop=True)
y_train=y[idx[:n_train]]
y_val=y[idx[n_train:n_train+n_val]]
y_test=y[idx[n_train+n_val:]]
#del df_train['fuel_efficiency_mpg']
#del df_val['fuel_efficiency_mpg']
#del df_test['fuel_efficiency_mpg']
df.isnull().sum()
df['horsepower'].median()


254.0

In [58]:
X_train_zero=df_train[base].fillna(0).values
X_val_zero=df_val[base].fillna(0).values
def train_linear(X,y,r=0):
    ones=np.ones(X.shape[0])
    X=np.column_stack([ones,X])
    XTX=X.T.dot(X)
    XTX=XTX+r*np.eye(XTX.shape[0]) 
    XTX_inv=np.linalg.inv(XTX)
    w_full=XTX_inv.dot(X.T).dot(y)
    return w_full[0],w_full[1:]
w0_zero,w_zero=train_linear(X_train_zero,y_train)
y_pred_zero=w0_zero+X_val_zero.dot(w_zero)
def rmes(y,y_pred):
    return np.sqrt(np.mean((y-y_pred)**2))
score_zero=rmes(y_val,y_pred_zero)
print(round(score_zero,3))

0.072


In [59]:
mean_hp=df_train['horsepower'].mean()
X_train=df_train[base].fillna(mean_hp).values
X_val=df_val[base].fillna(mean_hp).values
w0,w=train_linear(X_train,y_train)
y_pred=w0+X_val.dot(w)
score=rmes(y_val,y_pred)
print(round(score,3))

0.072


In [60]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [61]:
for r in[0.0,0.0001,0.001,0.01,0.1,1.0,10]:
    X_train=df_train[base].fillna(0).values
    w0,w=train_linear(X_train,y_train,r=r)
    X_val=df_val[base].fillna(0).values
    y_pred=w0+X_val.dot(w)
    score=rmes(y_val,y_pred)
    print(f'r{r:.4f}, RMSE={score:.4f}')

r0.0000, RMSE=0.0716
r0.0001, RMSE=0.0716
r0.0010, RMSE=0.0716
r0.0100, RMSE=0.0716
r0.1000, RMSE=0.0717
r1.0000, RMSE=0.0727
r10.0000, RMSE=0.0733


In [76]:
X_train_zero=df_train[base].fillna(0).values
X_val_zero=df_val[base].fillna(0).values
def train_linear(X,y,r=0.001):
    ones=np.ones(X.shape[0])
    X=np.column_stack([ones,X])
    XTX=X.T.dot(X)
    XTX=XTX+r*np.eye(XTX.shape[0]) 
    XTX_inv=np.linalg.inv(XTX)
    w_full=XTX_inv.dot(X.T).dot(y)
    return w_full[0],w_full[1:]
w0_zero,w_zero=train_linear(X_train_zero,y_train)
y_pred_zero=w0_zero+X_val_zero.dot(w_zero)
def rmes(y,y_pred):
    return np.sqrt(np.mean((y-y_pred)**2))
score_zero=rmes(y_val,y_pred_zero)
print(round(score_zero,3))

0.071


In [77]:
scores=[]
for seed in[0,1,2,3,4,5,6,7,8,9]:
    np.random.seed(seed)
    idx=np.arange(n)
    np.random.shuffle(idx)
    df_train=df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val=df.iloc[idx[n_train:n_train+n_val]].reset_index(drop=True)
    y_train=y[idx[:n_train]]
    y_val=y[idx[n_train:n_train+n_val]]
    X_train=df_train[base].fillna(0).values
    X_val=df_val[base].fillna(0).values
    w0,w=train_linear(X_train,y_train)
    y_pred=w0+X_val.dot(w)
    score=rmes(y_val,y_pred)
    scores.append(score)
print(scores)
print(round(np.std(scores),3))

[np.float64(0.07222155617176329), np.float64(0.07085565519484854), np.float64(0.07001627139184702), np.float64(0.07161015802810254), np.float64(0.07121587036948913), np.float64(0.0727219197385411), np.float64(0.07363953187092967), np.float64(0.07093675486336021), np.float64(0.07159390673561754), np.float64(0.07162797643033512)]
0.001


In [81]:
np.random.seed(42)
idx=np.arange(n)
np.random.shuffle(idx)
df_train=df.iloc[idx[:n_train]].reset_index(drop=True)
df_val=df.iloc[idx[n_train:n_train+n_val]].reset_index(drop=True)
df_test=df.iloc[idx[n_train+n_val:]].reset_index(drop=True)
y_train=y[idx[:n_train]]
y_val=y[idx[n_train:n_train+n_val]]
y_test=y[idx[n_train+n_val:]]
df_full_train=pd.concat([df_train,df_val]).reset_index(drop=True)
y_full_train=np.concatenate([y_train,y_val])
X_full_train=df_full_train[base].fillna(0).values
X_test=df_test[base].fillna(0).values
w0,w=train_linear(X_full_train,y_full_train,r=0.001)
y_pred=w0+X_test.dot(w)
print(round(rmes(y_test,y_pred),3))

0.072


In [79]:
print(df_test.columns)

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='object')
